# Grid Search — Baseline ResNet18

Systematic hyperparameter search over Phase 2 fine-tuning parameters for the baseline ResNet18 model under **3-fold stratified cross-validation**. Phase 1 is fixed based on prior validation that lr=1e-3 with no weight decay converges cleanly with the frozen backbone. Results are saved to CSV after every individual fold so the search can be safely interrupted and resumed without losing progress.

**Search space (9 configurations × 3 folds = 27 runs):**
- Phase 2 LR: `{1e-5, 5e-6, 1e-6}` — values starting with previous fine-tune rate and decreasing as mild overfitting was seen in pilot testing, so the search is focused on the range where regularisation and a reduced LR are more likely to interact beneficially
- Weight decay: `{0, 1e-4, 1e-3}` — covering no regularisation, light, and moderate AdamW decoupled regularisation

**Fixed parameters (not searched):**
- Phase 1 LR: `1e-3` — fixed; validated by tracking curves showing clean head convergence
- Phase 1 weight decay: `0` — head regularisation handled by Dropout(0.3); no additional decay needed
- Optimiser: AdamW (decoupled weight decay, Loshchilov & Hutter 2019)
- Criterion: BCEWithLogitsLoss
- Batch size: 32
- Phase 1 epochs: 20 (ceiling; early stopping patience=4 applies)
- Phase 2 epochs: 15 (ceiling; early stopping patience=3 applies)
- Architecture: base ResNet18, MLP head

**Selection criterion:** highest mean val accuracy across 3 folds. Ties broken by lowest mean val loss, then fewest Phase 2 epochs (via early stopping), then lowest weight decay.

## 1 · Paths & Imports

In [17]:
import sys
from pathlib import Path

ABSOLUTE_PATH = Path().resolve()
PROJECT_ROOT  = ABSOLUTE_PATH.parents[2]
DATA_DIR      = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR   = ABSOLUTE_PATH / "grid-search-results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(PROJECT_ROOT))

print(f"Project root : {PROJECT_ROOT}")
print(f"Data dir     : {DATA_DIR}")
print(f"Results dir  : {RESULTS_DIR}")

Project root : C:\Users\markm\Workspace\ms-machine-learning-diagnosis
Data dir     : C:\Users\markm\Workspace\ms-machine-learning-diagnosis\data\raw
Results dir  : C:\Users\markm\Workspace\ms-machine-learning-diagnosis\src\notebooks\CNNS\grid-search-results


In [18]:
import itertools
import csv
import os
import traceback
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import f1_score

import src.scripts.data    as data
import src.scripts.models  as models
import src.scripts.trainer as trainer
import src.scripts.utils   as utils

utils.set_seed(42)

Random seed set to 42 for Python, NumPy, and PyTorch


## 2 · Data — Outer Split (Identical to Main Experiment)

The 80/20 stratified outer split **must use the same seed as every other notebook** so that the held-out test set is identical across all stages. The grid search operates exclusively within `X_trainval / y_trainval`.

In [19]:
path, categories = data.get_dataset(str(DATA_DIR))
classes = data.get_classes(path, categories, visualise=False)

image_paths, labels = data.get_paths_and_labels(path, classes)
train_transform, test_transform = data.get_transforms()

# Outer split — fixed seed 42, never changes across stages
X_trainval, y_trainval, X_test, y_test = data.get_trainval_test_split(
    image_paths, labels,
    test_split=0.20,
    SEED=42
)

print(f"\nGrid search operates on {len(X_trainval)} train+val samples.")
print("Held-out test set is NOT used at any point in this notebook.")

get_dataset()>>> Dataset already exists in C:\Users\markm\Workspace\ms-machine-learning-diagnosis\data\raw
get_dataset()>>> Available categories: ['Control Axial_crop', 'Control Saggital_crop', 'MS Axial_crop', 'MS Saggital_crop']
get_paths_and_labels()>>> Total images: 1652
get_trainval_test_split()>>> Train+Val pool : 1321 (80.0%)
get_trainval_test_split()>>> Held-out test  : 331 (20.0%)
get_trainval_test_split()>>> TrainVal class ratio — MS: 520  Non-MS: 801
get_trainval_test_split()>>> Test     class ratio — MS: 130  Non-MS: 201

Grid search operates on 1321 train+val samples.
Held-out test set is NOT used at any point in this notebook.


## 3 · Grid Definition & Resume Logic

Results are written to `grid_search_results.csv` after **every individual fold**. On re-run, completed `(lr_p2, weight_decay, fold)` combinations are skipped automatically — a crash or kernel restart loses at most one fold.

In [20]:
def load_completed_runs(results_file):
    """Load set of (lr_p2, wd, fold) tuples already written to CSV."""
    completed = set()
    if results_file.exists():
        with open(results_file, newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                if row["error"] == "":
                    key = (
                        float(row["lr_phase2"]),
                        float(row["weight_decay"]),
                        int(row["fold"])
                    )
                    completed.add(key)
    return completed


def append_result(results_file, fieldnames, row_dict):
    """Append a single result row to CSV, creating header if file is new."""
    write_header = not results_file.exists()
    with open(results_file, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerow(row_dict)

In [21]:
# Search space — Phase 2 only; Phase 1 is fixed
# Expanded search space for Phase 2
GRID = {
    "lr_phase2"    : [5e-4, 1e-4, 5e-5],  # Increased from {1e-5, 5e-6, 1e-6}
    "weight_decay" : [1e-4, 1e-3, 1e-2],  # Shifted to include more aggressive 1e-2
}

# Fixed Phase 1 parameters (not searched)
LR_PHASE1   = 1e-3   # validated by prior tracking curves
WD_PHASE1   = 0      # head regularisation handled by Dropout(0.3)

# Fixed shared parameters
N_SPLITS      = 3    # 3-fold for grid search
BATCH_SIZE    = 32   # fixed — justified by Keskar et al. (2017)
P1_EPOCHS     = 20   # ceiling; early stopping applies
P2_EPOCHS     = 15   # ceiling; early stopping applies
P1_PATIENCE   = 4    # early stopping patience (Phase 1)
P2_PATIENCE   = 3    # early stopping patience (Phase 2)
SEED          = 42

RESULTS_FILE  = RESULTS_DIR / "grid_search_results.csv"
CSV_FIELDNAMES = [
    "lr_phase1", "wd_phase1", "lr_phase2", "weight_decay",
    "fold", "val_acc", "val_loss", "val_f1",
    "p1_epochs_run", "p2_epochs_run",
    "timestamp", "error"
]

# Generate all combinations
all_combos = list(itertools.product(
    GRID["lr_phase2"],
    GRID["weight_decay"]
))
total_runs = len(all_combos) * N_SPLITS
print(f"Grid: {len(all_combos)} configurations x {N_SPLITS} folds = {total_runs} total runs")
print(f"Fixed Phase 1: lr={LR_PHASE1:.0e}, weight_decay={WD_PHASE1}")

completed_runs = load_completed_runs(RESULTS_FILE)
print(f"Already completed: {len(completed_runs)} runs")

Grid: 9 configurations x 3 folds = 27 total runs
Fixed Phase 1: lr=1e-03, weight_decay=0
Already completed: 27 runs


## 4 · Grid Search Loop

Each run is wrapped in `try/except` — an OOM error or training failure logs the error and continues to the next combination. The CSV is written atomically after each fold so an interrupted session resumes cleanly.

**Expected duration:** ~20 min/run × 27 runs ≈ 9 hours. Start before an overnight session.

In [ ]:
run_number = len(completed_runs)

for lr_p2, wd in all_combos:
    for fold_idx in range(N_SPLITS):

        run_key = (lr_p2, wd, fold_idx)

        # Skip if already done
        if run_key in completed_runs:
            print(f"SKIP  lr2={lr_p2:.0e}  wd={wd:.0e}  fold={fold_idx+1}/{N_SPLITS}")
            continue

        utils.set_seed(SEED)

        run_number += 1
        print(f"\n{'='*65}")
        print(f"  Run {run_number}/{total_runs}  |  "
              f"lr2={lr_p2:.0e}  wd={wd:.0e}  fold={fold_idx+1}/{N_SPLITS}")
        print(f"{'='*65}")

        try:
            train_loader, val_loader = data.get_fold_loaders(
                X_trainval, y_trainval,
                fold_idx=fold_idx,
                train_transform=train_transform,
                test_transform=test_transform,
                n_splits=N_SPLITS,
                batch_size=BATCH_SIZE,
                SEED=SEED
            )

            model = models.get_model(architecture="base", head="mlp")

            run_configs = {
                "phase1": {
                    "num_epochs"  : P1_EPOCHS,
                    "lr"          : LR_PHASE1,       # fixed: 1e-3
                    "parameters"  : "head_and_attention",
                    "optimiser"   : optim.AdamW,
                    "criterion"   : nn.BCEWithLogitsLoss(),
                    "weight_decay": WD_PHASE1,       # fixed: 0
                },
                "phase2": {
                    "num_epochs"  : P2_EPOCHS,
                    "lr"          : lr_p2,            # searched
                    "parameters"  : "all",
                    "optimiser"   : optim.AdamW,
                    "criterion"   : nn.BCEWithLogitsLoss(),
                    "weight_decay": wd,               # searched
                },
            }

            # Phase 1: head warm-up (fixed config)
            _, _, _, val_accs_p1 = trainer.train_model(
                model, train_loader, val_loader,
                config_name="phase1",
                train_configs=run_configs,
                early_stopping_patience=P1_PATIENCE
            )
            p1_epochs_run = len(val_accs_p1)

            # Phase 2: full fine-tune (searched config)
            _, _, val_losses_p2, val_accs_p2 = trainer.train_model(
                model, train_loader, val_loader,
                config_name="phase2",
                train_configs=run_configs,
                early_stopping_patience=P2_PATIENCE
            )
            p2_epochs_run = len(val_accs_p2)

            # F1 inference pass on validation set
            model.eval()
            all_preds, all_labels = [], []
            with torch.no_grad():
                for imgs, lbls in val_loader:
                    imgs = imgs.to(next(model.parameters()).device)
                    logits = model(imgs).squeeze(1)
                    preds = (torch.sigmoid(logits) >= 0.5).long().cpu().tolist()
                    all_preds.extend(preds)
                    all_labels.extend(lbls.tolist())
            val_f1 = f1_score(all_labels, all_preds, average="binary", zero_division=0)

            result_row = {
                "lr_phase1"     : LR_PHASE1,
                "wd_phase1"     : WD_PHASE1,
                "lr_phase2"     : lr_p2,
                "weight_decay"  : wd,
                "fold"          : fold_idx,
                "val_acc"       : round(float(val_accs_p2[-1]),  6),
                "val_loss"      : round(float(val_losses_p2[-1]), 6),
                "val_f1"        : round(float(val_f1), 6),
                "p1_epochs_run" : p1_epochs_run,
                "p2_epochs_run" : p2_epochs_run,
                "timestamp"     : datetime.now().isoformat(timespec="seconds"),
                "error"         : "",
            }
            append_result(RESULTS_FILE, CSV_FIELDNAMES, result_row)
            completed_runs.add(run_key)

            print(f"val_acc={val_accs_p2[-1]:.4f}  "
                  f"val_f1={val_f1:.4f}  "
                  f"val_loss={val_losses_p2[-1]:.4f}  "
                  f"(p1={p1_epochs_run}ep  p2={p2_epochs_run}ep)")

        except Exception as e:
            error_msg = f"{type(e).__name__}: {str(e)}"
            print(f"ERROR -- {error_msg}")
            import traceback; traceback.print_exc()

            error_row = {
                "lr_phase1"     : LR_PHASE1,
                "wd_phase1"     : WD_PHASE1,
                "lr_phase2"     : lr_p2,
                "weight_decay"  : wd,
                "fold"          : fold_idx,
                "val_acc"       : "",
                "val_loss"      : "",
                "val_f1"        : "",
                "p1_epochs_run" : "",
                "p2_epochs_run" : "",
                "timestamp"     : datetime.now().isoformat(timespec="seconds"),
                "error"         : error_msg,
            }
            append_result(RESULTS_FILE, CSV_FIELDNAMES, error_row)

print(f"\n{'='*65}")
print("GRID SEARCH COMPLETE")
print(f"Results saved to: {RESULTS_FILE}")

## 5 · Results Analysis

Run this section after the search completes (or at any point during it to see partial results). Produces the full results table, the winning configuration, and heatmap visualisations.

In [22]:
# Load results
df_raw = pd.read_csv(RESULTS_FILE)

# Empty error fields are read as NaN by pandas — fill before filtering
df_raw["error"] = df_raw["error"].fillna("")

# Separate successful and failed runs
df_ok   = df_raw[df_raw["error"] == ""].copy()
df_fail = df_raw[df_raw["error"] != ""].copy()

df_ok["val_f1"]   = df_ok["val_f1"].astype(float)
df_ok["val_acc"]  = df_ok["val_acc"].astype(float)
df_ok["val_loss"] = df_ok["val_loss"].astype(float)

print(f"Successful runs : {len(df_ok)} / {total_runs}")
print(f"Failed runs     : {len(df_fail)}")
if len(df_fail) > 0:
    print("\nFailed configurations:")
    print(df_fail[["lr_phase1", "lr_phase2", "weight_decay", "fold", "error"]].to_string(index=False))

Successful runs : 27 / 27
Failed runs     : 0


In [23]:
df_ok

,lr_phase1,wd_phase1,lr_phase2,weight_decay,fold,val_acc,val_loss,val_f1,p1_epochs_run,p2_epochs_run,timestamp,error
0,0.001,0,0.000010,0.0000,0,0.918367,0.208508,0.885350,15,12,2026-03-12T12:46:12,
1,0.001,0,0.000010,0.0000,1,0.927273,0.198128,0.903614,20,15,2026-03-12T12:53:55,
2,0.001,0,0.000010,0.0000,2,0.888636,0.224960,0.867209,13,15,2026-03-12T13:00:08,
3,0.001,0,0.000010,0.0001,0,0.918367,0.208508,0.885350,15,12,2026-03-12T13:06:06,
4,0.001,0,0.000010,0.0001,1,0.927273,0.198128,0.903614,20,15,2026-03-12T13:13:48,
5,0.001,0,0.000010,0.0001,2,0.888636,0.224960,0.867209,13,15,2026-03-12T13:20:00,
6,0.001,0,0.000010,0.0010,0,0.918367,0.208508,0.885350,15,12,2026-03-12T13:25:57,
7,0.001,0,0.000010,0.0010,1,0.927273,0.198128,0.903614,20,15,2026-03-12T13:33:38,
8,0.001,0,0.000010,0.0010,2,0.888636,0.224960,0.867209,13,15,2026-03-12T13:39:51,
9,0.001,0,0.000005,0.0000,0,0.909297,0.214992,0.875000,15,15,2026-03-12T13:46:29,


In [24]:
# Aggregate across folds
group_cols = ["lr_phase2", "weight_decay"]

summary = (
    df_ok
    .groupby(group_cols)
    .agg(
        mean_val_f1   = ("val_f1",        "mean"),
        std_val_f1    = ("val_f1",        "std"),
        mean_val_acc  = ("val_acc",       "mean"),
        std_val_acc   = ("val_acc",       "std"),
        mean_val_loss = ("val_loss",      "mean"),
        std_val_loss  = ("val_loss",      "std"),
        mean_p2_epochs= ("p2_epochs_run", "mean"),
        n_folds       = ("val_f1",        "count"),
    )
    .reset_index()
    .sort_values(["mean_val_f1", "mean_val_loss"], ascending=[False, True])
)

# Format for display
summary["val_f1_fmt"]  = summary.apply(
    lambda r: f"{r.mean_val_f1:.4f} ± {r.std_val_f1:.4f}", axis=1)
summary["val_acc_fmt"] = summary.apply(
    lambda r: f"{r.mean_val_acc:.4f} ± {r.std_val_acc:.4f}", axis=1)
summary["val_loss_fmt"] = summary.apply(
    lambda r: f"{r.mean_val_loss:.4f} ± {r.std_val_loss:.4f}", axis=1)

display_cols = ["lr_phase2", "weight_decay",
                "val_f1_fmt", "val_acc_fmt", "val_loss_fmt",
                "mean_p2_epochs", "n_folds"]

print("\nFull results table (sorted by mean val macro-F1):\n")
summary[display_cols]


Full results table (sorted by mean val macro-F1):



,lr_phase2,weight_decay,val_f1_fmt,val_acc_fmt,val_loss_fmt,mean_p2_epochs,n_folds
6,0.000010,0.0000,0.8854 ± 0.0182,0.9114 ± 0.0202,0.2105 ± 0.0135,14.0,3
7,0.000010,0.0001,0.8854 ± 0.0182,0.9114 ± 0.0202,0.2105 ± 0.0135,14.0,3
8,0.000010,0.0010,0.8854 ± 0.0182,0.9114 ± 0.0202,0.2105 ± 0.0135,14.0,3
3,0.000005,0.0000,0.8805 ± 0.0149,0.9076 ± 0.0137,0.2215 ± 0.0146,15.0,3
4,0.000005,0.0001,0.8805 ± 0.0149,0.9076 ± 0.0137,0.2215 ± 0.0146,15.0,3
5,0.000005,0.0010,0.8805 ± 0.0149,0.9076 ± 0.0137,0.2215 ± 0.0146,15.0,3
0,0.000001,0.0000,0.8484 ± 0.0048,0.8819 ± 0.0040,0.2841 ± 0.0100,15.0,3
1,0.000001,0.0001,0.8484 ± 0.0048,0.8819 ± 0.0040,0.2841 ± 0.0100,15.0,3
2,0.000001,0.0010,0.8484 ± 0.0048,0.8819 ± 0.0040,0.2841 ± 0.0100,15.0,3


In [25]:
# Winning configuration
# Selection criterion:
#  1. Highest mean val F1 (primary — dataset is imbalanced)
#  2. Ties broken by highest mean val accuracy
#  3. Still tied: lowest mean val loss
#  4. Still tied: lowest weight decay

best = summary.sort_values(
    ["mean_val_f1", "mean_val_acc", "mean_val_loss", "weight_decay"],
    ascending=[False, False, True, True]
).iloc[0]

print("\n" + "="*55)
print("  WINNING CONFIGURATION")
print("="*55)
print(f"  Phase 2 LR   : {best.lr_phase2:.0e}")
print(f"  Weight decay : {best.weight_decay:.0e}")
print(f"  Mean val F1  : {best.mean_val_f1:.4f} ± {best.std_val_f1:.4f}")
print(f"  Mean val acc : {best.mean_val_acc:.4f} ± {best.std_val_acc:.4f}")
print(f"  Mean val loss: {best.mean_val_loss:.4f} ± {best.std_val_loss:.4f}")
print(f"  Mean p2 epochs (via early stopping): {best.mean_p2_epochs:.1f}")
print("="*55)

BEST_CONFIG = {
    "lr_phase1"    : 1e-3,             # fixed
    "lr_phase2"    : best.lr_phase2,
    "weight_decay" : best.weight_decay,
}
print(f"\nBEST_CONFIG = {BEST_CONFIG}")
print("\nCopy the BEST_CONFIG dict into the head ablation and architecture evaluation notebooks.")


  WINNING CONFIGURATION
  Phase 2 LR   : 1e-05
  Weight decay : 0e+00
  Mean val F1  : 0.8854 ± 0.0182
  Mean val acc : 0.9114 ± 0.0202
  Mean val loss: 0.2105 ± 0.0135
  Mean p2 epochs (via early stopping): 14.0

BEST_CONFIG = {'lr_phase1': 0.001, 'lr_phase2': np.float64(1e-05), 'weight_decay': np.float64(0.0)}

Copy the BEST_CONFIG dict into the head ablation and architecture evaluation notebooks.


In [ ]:
# Save summary table to CSV
summary_path = RESULTS_DIR / "grid_search_summary.csv"
summary[display_cols].to_csv(summary_path, index=False)
print(f"Summary table saved to: {summary_path}")

## 6 · Visualisation

Three heatmaps (one per weight decay value) showing mean val accuracy across the Phase 1 LR × Phase 2 LR grid. Reveals whether LR sensitivity is uniform across regularisation strengths.

In [ ]:
lr2_values = sorted(summary["lr_phase2"].unique(), reverse=True)
wd_values  = sorted(summary["weight_decay"].unique())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 heatmap (primary metric)
pivot_f1 = summary.pivot(index="lr_phase2", columns="weight_decay", values="mean_val_f1")
pivot_f1 = pivot_f1.reindex(index=lr2_values, columns=wd_values)
sns.heatmap(pivot_f1, ax=axes[0], annot=True, fmt=".4f", cmap="YlGn",
            linewidths=0.5, cbar=True)
axes[0].set_title("Mean Val F1: Phase 2 LR x Weight Decay", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Weight Decay", fontsize=11)
axes[0].set_ylabel("Phase 2 LR", fontsize=11)

# Accuracy heatmap (secondary)
pivot_acc = summary.pivot(index="lr_phase2", columns="weight_decay", values="mean_val_acc")
pivot_acc = pivot_acc.reindex(index=lr2_values, columns=wd_values)
sns.heatmap(pivot_acc, ax=axes[1], annot=True, fmt=".4f", cmap="YlGn",
            linewidths=0.5, cbar=True)
axes[1].set_title("Mean Val Accuracy: Phase 2 LR x Weight Decay", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Weight Decay", fontsize=11)
axes[1].set_ylabel("Phase 2 LR", fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Std heatmap — stability across folds
pivot_std = summary.pivot(index="lr_phase2", columns="weight_decay", values="std_val_acc")
pivot_std = pivot_std.reindex(index=lr2_values, columns=wd_values)

fig2, ax2 = plt.subplots(figsize=(8, 5))
sns.heatmap(
    pivot_std,
    ax=ax2,
    annot=True, fmt=".4f",
    cmap="YlOrRd_r",
    linewidths=0.5,
    cbar=True
)
ax2.set_title("Val Accuracy Std: Phase 2 LR x Weight Decay", fontsize=13, fontweight="bold")
ax2.set_xlabel("Weight Decay", fontsize=11)
ax2.set_ylabel("Phase 2 LR", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Per-fold breakdown for the winning config
# Sanity check: verify the winning config's fold results individually
# to confirm the mean isn't being driven by a single lucky fold.

best_rows = df_ok[
    (df_ok["lr_phase2"]    == best.lr_phase2) &
    (df_ok["weight_decay"] == best.weight_decay)
].sort_values("fold")

print("Per-fold results for winning configuration:")
print(f"  lr_p2={best.lr_phase2:.0e}  "
      f"wd={best.weight_decay:.0e}\n")

for _, row in best_rows.iterrows():
    print(f"  Fold {int(row.fold)+1}: "
          f"val_f1={row.val_f1:.4f}  "
          f"val_acc={row.val_acc:.4f}  "
          f"val_loss={row.val_loss:.4f}  "
          f"(p1={int(row.p1_epochs_run)}ep  "
          f"p2={int(row.p2_epochs_run)}ep)")

print(f"\n  Mean F1  : {best_rows.val_f1.mean():.4f}")
print(f"  Std F1   : {best_rows.val_f1.std():.4f}")
print(f"  Mean Acc : {best_rows.val_acc.mean():.4f}")
print(f"  Std Acc  : {best_rows.val_acc.std():.4f}")

## 7 · Next Steps

1. **Record `BEST_CONFIG`** from Cell 5 above — this dict is your fixed protocol for all subsequent notebooks.
2. **Head ablation (Stage 3):** Run `head-ablation.ipynb` using `BEST_CONFIG`. Determines whether MLP or linear probe head is optimal. Uses 5-fold CV.
3. **Architecture evaluation (Stage 4):** Run all 8 architecture variants under 5-fold CV using `BEST_CONFIG` and the winning head type from Stage 3.
4. The `grid_search_summary.csv` file in `grid-search-results/` contains the full 27-row results table for your dissertation appendix.